# 9.a — Ad hoc test: local novelty with the sub-grid relief

**Ad hoc test, nothing of the chain is changed**: it reads the training tables and final models of 3.g, the outputs of
4.g and the relief of 0.5, and writes only in `<out_root>/test_relief_novelty/<route>/<set>/`.

**Question.** A part of the land that the SVM calls suitable is removed by the ECOCROP TMIN (cold tail): flat cold
cells (taiga) whose mean climate is that of cold mountain presences (the crop grows in the valleys). The combination
"cold + flat" is absent from the training cells, but the MESS (variable by variable) and NT2 (global Mahalanobis
ellipse) do not see it, and the relief is not a predictor. Would a **local** novelty check in the space
*climate + relief* flag these cells as "climate never seen in training", without changing the SVM or the data?

**Method (fixed before looking at the results).** Space: the 10 predictors standardised with the scaler of the final
model, plus RELIEF (0.5) standardised on the training cells (for "climate + relief"); the climate-only space is the
control. For every land cell: Euclidean distance to the nearest training cell (presence or absence). Threshold: the
95th or 99th percentile of the distances of the training cells to their nearest other training cell. A cell is flagged
when its distance exceeds the threshold.

**Compared**, per crop: on ERA5, the land flagged beyond the current novelty, its share in the land removed by the TMIN
(the cold tail), north of 50° N outside the cropland, and in today's crop area (should be ~0); in the four runs, the
share of today's crop area and of the climatically suitable land flagged (the pitfall of the AOA, which flagged 16–64 %
of the 2050 crop area). Maps for maize.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PCTS = [95, 99]                   # thresholds: percentiles of the nearest-neighbour distances of the training cells

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
OUT = cu.out_dir(CFG, "test_relief_novelty", ROUTE, SET)          # the only folder written
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
ANY = cu.mirca_cropland(CFG).values
LAT2 = np.broadcast_to(GRID["lat"][:, None], LAND.shape)
N50_OUT = LAND & ~ANY & (LAT2 > 50)
REL = cu.relief(CFG).values.astype(float)
mha = lambda m: float(np.nansum(LAND_MHA * m))
print(f"set {SET} | written only in {OUT}")

## 1. Nearest-neighbour distances and flags

In [ ]:
def bio_file(scenario, crop):
    if scenario == "ERA5":
        return ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
    return DELTA / f"BIOvariables_delta_{scenario}_{crop}.nc"


SPACES = ["climate", "climate + relief"]
rows, MAPS = [], {}
for crop in CROPS:
    b = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, sc = b["bio_cols"], b["scaler"]
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(bio_file("ERA5", crop)) as ds:
        A0 = cu.bio_array(ds, cols, CFG)
    Xtr = A0[ii, jj]
    ok_tr = np.isfinite(Xtr).all(axis=1) & np.isfinite(REL[ii, jj])
    r_mu, r_sd = REL[ii, jj][ok_tr].mean(), REL[ii, jj][ok_tr].std()
    def embed(X, rel, space):
        Z = sc.transform(X)
        return np.column_stack([Z, (rel - r_mu) / r_sd]) if space == "climate + relief" else Z
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        NOV = {s: ds["novel"].sel(scenario=s).values == 1 for s in SCEN}
        PRB = {s: np.nan_to_num(ds["suit_proba"].sel(scenario=s).values) for s in SCEN}
        COLD = {s: ds["below_tmin"].sel(scenario=s).values == 1 for s in SCEN}
        CLIM = {s: ds["suit_bin"].sel(scenario=s).values == 1 for s in SCEN}
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    tmin_removed = {s: LAND & (PRB[s] >= cu.CLIMATE_P) & COLD[s] & ~NOV[s] for s in SCEN}   # the cold tail
    for space in SPACES:
        Ztr = embed(Xtr[ok_tr], REL[ii, jj][ok_tr], space)
        nn = NearestNeighbors(n_neighbors=2).fit(Ztr)
        d_tr = nn.kneighbors(Ztr)[0][:, 1]                      # to the nearest OTHER training cell
        thr = {p: float(np.percentile(d_tr, p)) for p in PCTS}
        for s in SCEN:
            if s == "ERA5":
                A = A0
            else:
                with xr.open_dataset(bio_file(s, crop)) as ds:
                    A = cu.bio_array(ds, cols, CFG)
            ok = LAND & np.isfinite(A).all(axis=-1) & np.isfinite(REL)
            d = np.full(LAND.shape, np.nan)
            d[ok] = nn.kneighbors(embed(A[ok], REL[ok], space), n_neighbors=1)[0][:, 0]
            for p in PCTS:
                flag = ok & (d > thr[p])
                new = flag & ~NOV[s]
                MAPS[(crop, space, p, s)] = new
                rows.append({"crop": crop, "space": space, "percentile": p, "scenario": s, "threshold": thr[p],
                             "flagged beyond current novelty (Mha)": mha(new),
                             "of the TMIN-removed land flagged (%)": 100 * mha(new & tmin_removed[s]) / max(mha(tmin_removed[s]), 1e-9),
                             "TMIN-removed land (Mha)": mha(tmin_removed[s]),
                             "N of 50 outside cropland flagged (Mha)": mha(new & N50_OUT),
                             "today's crop area flagged (%)": share(new),
                             "climatically suitable land flagged (%)": 100 * mha(new & CLIM[s]) / max(mha(CLIM[s]), 1e-9)})
    print(f"{crop}: done", flush=True)
res = pd.DataFrame(rows).set_index(["crop", "space", "percentile", "scenario"])
res.to_csv(OUT / f"relief_novelty_{SET}.csv")
with pd.option_context("display.max_rows", 400, "display.max_columns", 30, "display.width", 250):
    print("(1) ERA5"); display(res.xs("ERA5", level="scenario").round(2))
    print("(2) the four runs: today's crop area flagged (%)")
    display(res["today's crop area flagged (%)"].drop("ERA5", level="scenario").unstack("scenario").round(2))
    print("(3) the four runs: climatically suitable land flagged (%)")
    display(res["climatically suitable land flagged (%)"].drop("ERA5", level="scenario").unstack("scenario").round(2))

## 2. Maps, maize: ERA5 and 2050 no hosing, climate + relief

In [ ]:
cols4 = (["#e6e6e6", "#fdae61", "#2c7bb6", "#7b3294", "#7f7f7f"],
         ["neither", "removed by TMIN only", "flagged by the relief novelty only", "both", "already novel (current)"])
crop = "Maize_rf" if "Maize_rf" in CROPS else CROPS[0]
with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
    NOV = {s: ds["novel"].sel(scenario=s).values == 1 for s in SCEN}
    PRB = {s: np.nan_to_num(ds["suit_proba"].sel(scenario=s).values) for s in SCEN}
    COLD = {s: ds["below_tmin"].sel(scenario=s).values == 1 for s in SCEN}
show = [s for s in ("ERA5", "gcm_2050_nohosing") if s in SCEN]
fig, axes = plt.subplots(len(PCTS), len(show), figsize=(12 * len(show), 5.4 * len(PCTS)), squeeze=False)
for r, p in enumerate(PCTS):
    for k, s in enumerate(show):
        tm = LAND & (PRB[s] >= cu.CLIMATE_P) & COLD[s] & ~NOV[s]
        fl = MAPS[(crop, "climate + relief", p, s)]
        c = np.where(NOV[s], 4.0, np.where(tm & fl, 3.0, np.where(fl, 2.0, np.where(tm, 1.0, 0.0))))
        ax = axes[r, k]
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, c, np.nan), vmin=-0.5, vmax=4.5, cmap=ListedColormap(cols4[0]), shading="nearest")
        ax.set_ylim(-60, 85); ax.set_title(f"{crop}, {s}, climate + relief, p{p}: flagged {mha(fl):.0f} Mha", fontsize=11)
        ax.legend(handles=[Patch(color=a, label=l) for a, l in zip(*cols4)], loc="lower left", fontsize=8, framealpha=0.9)
fig.savefig(OUT / f"relief_novelty_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()